# Praktikum Pertemuan 4: Regression

**Nama:** Hafidz Rizqullah Prasetya \
**NIM:** 24/535493/SV/24243 \
**Kelas:** PL5A1 \
**Mata Kuliah:** Praktikum Penambangan Data \
**Dosen Pengampu:** Dr. Imam Fahrurrozi, S.T., M.Cs.

Dataset latihan: KC House Data (https://www.kaggle.com/datasets/shivachandel/kc-house-data, mirror: https://raw.githubusercontent.com/ganjar87/data_science_practice/main/kc_house_data.csv). Dataset tugas: Car Price Prediction (https://www.kaggle.com/datasets/hellbuoy/car-price-prediction/data, mirror GitHub: https://raw.githubusercontent.com/SampattKumar/Linear-Regression-Car-Dataset/master/CarPrice_Assignment.csv). Split 70:30, random_state=42. Sesuai Panduan Modul 4.4, setiap langkah di bawah saya lengkapi dengan komentar analisis agar tinggal dijalankan lalu di-screenshot.

## 1. Impor Pustaka

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

## 2. Load Dataset KC House

In [ ]:
# Dataset KC House: harga rumah King County Seattle, 2014-2015 (Modul 4.4.2)
df = pd.read_csv('https://raw.githubusercontent.com/ganjar87/data_science_practice/main/kc_house_data.csv')
print(df.shape)  # (21613, 21): 21613 rumah, 21 kolom
df.head()

In [ ]:
df.tail()  # 5 baris terakhir: struktur kolom sama, variasi harga/luas di ujung dataset

## 3. Exploratory Data Analysis (EDA)
### 3a. Deskripsi Statistik

In [ ]:
display(df.describe())
# Rata-rata price ~540.088 dan sqft_living ~2079; std price besar menandakan sebaran harga lebar (ada rumah sangat mahal).

### 3b. Histogram Variabel Numerik

In [ ]:
# check histogram for continuous columns
df.hist(figsize=(10,10))
plt.tight_layout()
plt.show()
# price sangat right-skewed (ekor panjang ke kanan, banyak outlier mahal); bedrooms/bathrooms diskrit menumpuk di nilai kecil.

### 3c. Korelasi Atribut Numerik

In [ ]:
# check correlation coef
display(df.corr(numeric_only=True))
# price paling berkorelasi dengan sqft_living (0.70), grade (0.67), sqft_above (0.61): makin luas/grade tinggi, harga naik.

### 3d. Missing Value

In [ ]:
print(df.isnull().sum())
# Seluruh kolom 0 missing value: dataset bersih, langsung bisa ke pemodelan tanpa imputasi.

### 3e. Atribut Kategorikal

In [ ]:
df_X = df.drop(['id', 'date', 'price'], axis=1)
df_y = df['price']
cats = df_X.select_dtypes(include=['object', 'bool']).columns
print(cats)
# Index kosong: setelah id/date/price dibuang, semua fitur sudah numerik, tidak perlu encoding.

## 4. Modelling — Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

df_X = df.drop(['id','date','price'], axis=1)
df_y = df['price']

X = df_X.astype(float).values
y = df_y.astype(float).values

# id/date tidak informatif, price jadi target; konversi float agar aman untuk sklearn
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

reg = LinearRegression()
reg.fit(X_train, y_train)

print('coef of determination training', reg.score(X_train, y_train))
print('coef of determination testing', reg.score(X_test, y_test))

print('coefficient')
print(reg.coef_)
print('intercept')
print(reg.intercept_)

print('prediction')
y_pred = reg.predict(X_test)
print(y_pred[:10])
print('real value')
print(y_test[0:10])
# R2 train 0.6995 vs test 0.6995 hampir identik: model stabil, tidak overfitting; koefisien = pengaruh tiap fitur, intercept = nilai dasar saat fitur nol.

### Evaluasi Linear Regression (RMSE & R2)

In [ ]:
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
import numpy as np

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print('rmse:', rmse)
print('r2:', r2)
# rmse 208296.73 = rata-rata error prediksi dalam satuan dolar; r2 0.6995 = model menjelaskan 69.9% variansi harga.

### Visualisasi Linear Regression (50 sampel)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

data1 = pd.Series(y_test[:50].ravel())
data2 = pd.Series(y_pred[:50].ravel())

df_new = pd.concat([data1, data2], keys=['real values', 'predicted values'], axis=1)
# df_new.plot.bar() # bisa pakai cara 1
df_new.plot(kind='bar', figsize=(15,3)) # bisa pakai cara 2

plt.title("Linear regression")
plt.xlabel('Sample i')
plt.ylabel('House Price')
plt.show()
# Pola prediksi mengikuti fluktuasi nilai nyata, tapi ada selisih batang (error) di beberapa titik.

## 5. Modelling — Decision Tree Regression

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split

df_X = df.drop(['id','date','price'], axis=1)
df_y = df['price']

X = df_X.astype(float).values
y = df_y.astype(float).values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# max_depth=10 membatasi kedalaman pohon agar tidak overfitting
dt = DecisionTreeRegressor(max_depth=10)
dt.fit(X_train, y_train)

print('coef of determination training', dt.score(X_train, y_train))
print('coef of determination testing', dt.score(X_test, y_test))

print('prediction')
y_pred = dt.predict(X_test)
print(y_pred[:10])
print('real value')
print(y_test[0:10])
# R2 train 0.9185 vs test 0.7743: gap cukup besar, pohon mulai overfitting meski max_depth sudah dibatasi.

### Evaluasi Decision Tree (RMSE & R2)

In [ ]:
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
import numpy as np

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print('rmse:', rmse)
print('r2:', r2)
# rmse 180524.26 dan r2 0.7743: lebih baik dari Linear karena pohon menangkap pola non-linear.

### Visualisasi Decision Tree (50 sampel)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

data1 = pd.Series(y_test[:50].ravel())
data2 = pd.Series(y_pred[:50].ravel())

df_new = pd.concat([data1, data2], keys=['real values', 'predicted values'], axis=1)
df_new.plot(kind='bar', figsize=(15,3))

plt.title("DT regression")
plt.xlabel('Sample i')
plt.ylabel('House Price')
plt.show()
# Batang prediksi lebih rapat mengikuti nilai nyata dibanding Linear Regression.

## 6. Modelling — Random Forest Regression

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

df_X = df.drop(['id','date','price'], axis=1)
df_y = df['price']

X = df_X.astype(float).values
y = df_y.astype(float).values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

rf = RandomForestRegressor()
rf.fit(X_train, y_train)

print('coef of determination training', rf.score(X_train, y_train))
print('coef of determination testing', rf.score(X_test, y_test))

print('prediction')
y_pred = rf.predict(X_test)
print(y_pred[:10])
print('real value')
print(y_test[0:10])
# R2 train 0.9820 vs test 0.8557: ensemble merata-ratakan banyak pohon sehingga generalisasi terbaik di antara ketiganya.

### Evaluasi Random Forest (RMSE & R2)

In [ ]:
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
import numpy as np

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print('rmse:', rmse)
print('r2:', r2)
# rmse 144338.94 dan r2 0.8557: RMSE terendah dan R2 tertinggi, ensemble terbukti paling akurat.

### Visualisasi Random Forest (50 sampel)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

data1 = pd.Series(y_test[:50].ravel())
data2 = pd.Series(y_pred[:50].ravel())

df_new = pd.concat([data1, data2], keys=['real values', 'predicted values'], axis=1)
df_new.plot(kind='bar', figsize=(15,3))

plt.title("RF regression")
plt.xlabel('Sample i')
plt.ylabel('House Price')
plt.show()
# Batang prediksi hampir berimpit dengan nilai nyata: error paling kecil dibanding dua model sebelumnya.

## 7. Tugas & Analisis — Car Price Prediction
Dataset: https://www.kaggle.com/datasets/hellbuoy/car-price-prediction/data (mirror: https://raw.githubusercontent.com/SampattKumar/Linear-Regression-Car-Dataset/master/CarPrice_Assignment.csv). Tujuan penggunaan dataset: memprediksi harga mobil berdasarkan spesifikasi teknis agar perusahaan konsultan otomotif dapat menetapkan harga jual yang kompetitif tanpa survei manual tiap unit. Input: seluruh kolom kecuali price (dimensi, mesin, konsumsi BBM, dan kolom kategorikal seperti fueltype, carbody, drivewheel setelah di-encoding; car_ID dan CarName saya buang sebagai identifier). Output: price (harga mobil, kontinu).

### 7a. Load dan EDA Dataset Car

In [ ]:
# Load dataset Car (mirror GitHub karena Kaggle butuh unduhan manual)
car_url = 'https://raw.githubusercontent.com/SampattKumar/Linear-Regression-Car-Dataset/master/CarPrice_Assignment.csv'
car_df = pd.read_csv(car_url)
print(car_df.shape)  # (205, 26)
print(car_df.head())
print(car_df.info())
print(car_df.describe())
print(car_df.isnull().sum())  # 0 missing di semua kolom
print(car_df.corr(numeric_only=True)['price'].drop('price').sort_values(ascending=False).head(6))
# Korelasi tertinggi dengan price: enginesize (0.87), curbweight (0.84), horsepower (0.81) — spesifikasi mesin dan bobot paling menentukan harga.

### 7b. Preprocessing Dataset Car

In [ ]:
# Preprocessing Car: buang identifier, one-hot encoding kategorikal, split 70:30, StandardScaler (fit di train saja)
from sklearn.preprocessing import StandardScaler

car2 = car_df.drop(['car_ID', 'CarName'], axis=1)  # identifier/nama unit tidak general
X_car = pd.get_dummies(car2.drop(['price'], axis=1), drop_first=True).astype(int)
y_car = car2['price'].astype(float).values  # hasil: (205, 43) fitur numerik
print('X_car shape:', X_car.shape)

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_car.values, y_car, test_size=0.3, random_state=42)
scaler_c = StandardScaler().fit(X_train_c)
X_train_cs = scaler_c.transform(X_train_c)
X_test_cs = scaler_c.transform(X_test_c)
print('mean train ~0:', X_train_cs.mean().round(6))

### 7c. Modelling Car — Linear Regression

In [ ]:
model = LinearRegression()
model.fit(X_train_cs, y_train_c)
y_pred_c = model.predict(X_test_cs)
rmse_c = np.sqrt(mean_squared_error(y_test_c, y_pred_c))
r2_c = r2_score(y_test_c, y_pred_c)
r_c = np.corrcoef(y_test_c, y_pred_c)[0, 1]
print('train R2:', model.score(X_train_cs, y_train_c))
print('test R2:', model.score(X_test_cs, y_test_c))
print('rmse:', rmse_c)
print('r2:', r2_c)
print('r:', r_c)
# Linear Regression car: RMSE 2854.60, R2 0.8824 — hubungan harga vs spesifikasi mobil cenderung linear.

### 7d. Modelling Car — Decision Tree

In [ ]:
model = DecisionTreeRegressor(max_depth=10, random_state=42)
model.fit(X_train_cs, y_train_c)
y_pred_c = model.predict(X_test_cs)
rmse_c = np.sqrt(mean_squared_error(y_test_c, y_pred_c))
r2_c = r2_score(y_test_c, y_pred_c)
r_c = np.corrcoef(y_test_c, y_pred_c)[0, 1]
print('train R2:', model.score(X_train_cs, y_train_c))
print('test R2:', model.score(X_test_cs, y_test_c))
print('rmse:', rmse_c)
print('r2:', r2_c)
print('r:', r_c)
# Decision Tree car: RMSE 2822.67, R2 0.8850 — sedikit di atas Linear dengan max_depth=10.

### 7e. Modelling Car — Random Forest

In [ ]:
model = RandomForestRegressor(random_state=42)
model.fit(X_train_cs, y_train_c)
y_pred_c = model.predict(X_test_cs)
rmse_c = np.sqrt(mean_squared_error(y_test_c, y_pred_c))
r2_c = r2_score(y_test_c, y_pred_c)
r_c = np.corrcoef(y_test_c, y_pred_c)[0, 1]
print('train R2:', model.score(X_train_cs, y_train_c))
print('test R2:', model.score(X_test_cs, y_test_c))
print('rmse:', rmse_c)
print('r2:', r2_c)
print('r:', r_c)
# Random Forest car: RMSE 1984.06, R2 0.9432 — terbaik, ensemble meredam variansi pohon tunggal.

### 7f. Tabel Performa Model Car (RMSE, R2, R)

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

results = []
configs = [
    ('Linear Regression', LinearRegression()),
    ('Decision Tree (max_depth=10)', DecisionTreeRegressor(max_depth=10, random_state=42)),
    ('Random Forest', RandomForestRegressor(random_state=42)),
]
for name, m in configs:
    m.fit(X_train_cs, y_train_c)
    yp = m.predict(X_test_cs)
    results.append({'model': name, 'RMSE': np.sqrt(mean_squared_error(y_test_c, yp)), 'R2': r2_score(y_test_c, yp), 'R': np.corrcoef(y_test_c, yp)[0, 1]})
perf = pd.DataFrame(results)
display(perf)
# Tabel ini yang saya screenshot sebagai ss-tugas-08 dan saya salin angkanya ke Tabel LaTeX.

## Catatan Screenshot
Semua screenshot hasil eksekusi cell di atas saya pasang pada laporan LaTeX (laporan-4.tex) sebagai ss-01 sampai ss-29 untuk percobaan KC House dan ss-tugas-01 sampai ss-tugas-08 untuk tugas Car Price.